In [43]:
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import os
sns.set_theme()
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
cwd = os.getcwd()
print("Current working directory:", cwd)

# Change the working directory
#os.chdir('S:/OneDrive - University of Georgia/1 UGA/1 PhD/1 Spring24/Tobacco')
os.chdir(r'/Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/1 Spring24/Tobacco')

# Verify the change
print("New working directory:", os.getcwd())

Current working directory: /Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/1 Spring24/Tobacco
New working directory: /Users/sus/Library/CloudStorage/OneDrive-UniversityofGeorgia/1 UGA/1 PhD/1 Spring24/Tobacco


In [44]:
# Loading the Smoke and Heat dataset

df1 = pd.read_csv('cut_market2.csv')
smoke_market1_df_shape = df1.shape
df1['type'] = 1
df2 = pd.read_csv('smokeless_market2.csv') 
heat_market1_df_shape = df2.shape
df2['type'] = 2




# Analyse the Smoke dataset

Rows and Columns

In [45]:

total_rows = len(df1)
print(f"Total number of rows: {total_rows}")

# List the column names
print(df1.columns)

Total number of rows: 4652
Index(['sku', 'sku_id', 'brand_n', 'multi', 'size', 'W', 'totrev', 'qt',
       'price', 'region', 'type'],
      dtype='object')


Unique identifier

In [46]:
unique_counts = df1.nunique()
uniqueness_ratio = (unique_counts / total_rows) * 100

# Print columns that have 100% uniqueness
potential_ids = uniqueness_ratio[uniqueness_ratio == 100].index.tolist()

print("\n--- Uniqueness Report ---")
print(uniqueness_ratio.sort_values(ascending=False).to_string())

if potential_ids:
    print(f"\nPotential Unique Identifier(s) (100% Unique): **{potential_ids}**")
else:
    print("\nNo single column is 100% unique.")


--- Uniqueness Report ---
totrev     7.695615
W          3.933792
qt         2.858985
region     0.279450
sku        0.128977
sku_id     0.128977
brand_n    0.128977
price      0.107481
multi      0.021496
size       0.021496
type       0.021496

No single column is 100% unique.


In [47]:
# 1. Define the candidates
cols_to_check = ['brand_n', 'region', 'W']

# 2. Check for uniqueness
# Group by these 3 columns and count size
duplicate_check = df1.groupby(cols_to_check).size()

# 3. Print results
print(f"Checking combination: {cols_to_check}")
if duplicate_check.max() == 1:
    print("✅ Success! ['brand_n', 'region', 'W'] is the unique identifier.")
else:
    print(f"❌ Not unique. The maximum number of repeats for a combination is {duplicate_check.max()}.")
    
    # Optional: See which ones are duplicated
    print("\nSample duplicates:")
    print(df1[df1.duplicated(subset=cols_to_check, keep=False)].sort_values(by=cols_to_check).head())

Checking combination: ['brand_n', 'region', 'W']
✅ Success! ['brand_n', 'region', 'W'] is the unique identifier.


Variables with missing data

In [48]:
# Checking for missing variables
missing_counts = df1.isnull().sum()
print("--- Count of Missing Values Per Column ---")
print(missing_counts)

# Filter the series to only include counts > 0
missing_data_columns = missing_counts[missing_counts > 0].sort_values(ascending=False)

if missing_data_columns.empty:
    print("\n✅ Great news! No missing values found in your entire dataset.")
else:
    print("\n--- Columns with Missing Values (Count) ---")
    print(missing_data_columns.to_string())

--- Count of Missing Values Per Column ---
sku           0
sku_id        0
brand_n       0
multi         0
size          0
W             0
totrev     3854
qt         3854
price      3854
region        0
type          0
dtype: int64

--- Columns with Missing Values (Count) ---
totrev    3854
qt        3854
price     3854


Making the panel balanced

In [49]:
# 1. Get the unique count for each key variable
num_brands = df1['brand_n'].nunique()
num_regions = df1['region'].nunique()
num_weeks = df1['W'].nunique()

# 2. Calculate the expected number of rows for a balanced panel
expected_rows = num_brands * num_regions * num_weeks

print(f"Unique Brands: {num_brands}")
print(f"Unique Regions: {num_regions}")
print(f"Unique Weeks (W): {num_weeks}")
print(f"Expected Rows for a BALANCED panel: {expected_rows}")

actual_rows = len(df1)
print(f"Actual Rows in df1: {actual_rows}")

if actual_rows == expected_rows:
    print("\n✅ The panel is **BALANCED**. Every brand-region combination has data for every week.")
else:
    missing_rows = expected_rows - actual_rows
    print(f"\n❌ The panel is **UNBALANCED**. There are {missing_rows} missing brand-region-week combinations.")
    print("The actual number of rows is less than the expected number.")

Unique Brands: 6
Unique Regions: 13
Unique Weeks (W): 183
Expected Rows for a BALANCED panel: 14274
Actual Rows in df1: 4652

❌ The panel is **UNBALANCED**. There are 9622 missing brand-region-week combinations.
The actual number of rows is less than the expected number.


In [50]:
import pandas as pd
from itertools import product

if actual_rows != expected_rows:
    
    # 1. Get all unique values for each key
    all_brands = df1['brand_n'].unique()
    all_regions = df1['region'].unique()
    all_weeks = df1['W'].unique()

    # 2. Create a DataFrame of ALL POSSIBLE combinations (the full panel structure)
    full_index = pd.DataFrame(
        product(all_brands, all_regions, all_weeks),
        columns=['brand_n', 'region', 'W']
    )

    # 3. Merge the full index with the actual data (using a left join)
    # The 'indicator=True' option tells us which rows only exist in the full_index
    merged_df = full_index.merge(
        df1,
        on=['brand_n', 'region', 'W'],
        how='left',
        indicator=True
    )

    # 4. Filter for rows that are missing in the original data ('left_only')
    missing_combinations = merged_df[merged_df['_merge'] == 'left_only'][['brand_n', 'region', 'W']]

    print("\n--- Missing Combinations Sample ---")
    if not missing_combinations.empty:
        # Display the first few missing combinations
        print(f"Total missing combinations found: {len(missing_combinations)}")
        print(missing_combinations.head())
    else:
        print("No missing combinations found (This should match the initial check).")


--- Missing Combinations Sample ---
Total missing combinations found: 9622
     brand_n  region  W
183      428       2  1
184      428       2  2
185      428       2  3
186      428       2  4
187      428       2  5


We donot have balanced panel, lets make one:

In [51]:

# 1. AGGREGATE DATA (With NaN protection)
# We use a lambda function to pass 'min_count=1'. 
# This ensures that if the original data was NaN, the sum remains NaN (instead of becoming 0).
df_aggregated = df1.groupby(['brand_n', 'region', 'W']).agg({
    'qt': lambda x: x.sum(min_count=1),      
    'totrev': lambda x: x.sum(min_count=1),  
    'price': 'mean', 
    'type': 'min'  
}).reset_index()

# 2. DEFINE THE SKELETON
unique_brands = df_aggregated['brand_n'].unique()
unique_regions = df_aggregated['region'].unique()
unique_weeks = df_aggregated['W'].unique()

full_index = pd.MultiIndex.from_product(
    [unique_brands, unique_regions, unique_weeks], 
    names=['brand_n', 'region', 'W']
)

# 3. REINDEX (Add the new missing rows)
# Original rows keep their data (including original NaNs preserved above).
# New rows are created as NaN.
df1_balanced = df_aggregated.set_index(['brand_n', 'region', 'W']).reindex(full_index).reset_index()

# 4. CLEANUP
cols_to_drop = ['sku', 'sku_id', 'multi', 'size']
df1_balanced = df1_balanced.drop(columns=cols_to_drop, errors='ignore')

# Fill only 'type' as requested
df1_balanced['type'] = df1_balanced['type'].fillna(1)

# --- VERIFICATION ---
print(f"Final Row Count: {len(df1_balanced)}")
print("\n--- Missing Value Counts (Includes both original and new NaNs) ---")
print(df1_balanced[['qt', 'totrev', 'price']].isnull().sum())

Final Row Count: 14274

--- Missing Value Counts (Includes both original and new NaNs) ---
qt        13476
totrev    13476
price     13476
dtype: int64


# Analyse the Smoke dataset
Rows and Columns

In [52]:
total_rows = len(df2)
print(f"Total number of rows: {total_rows}")

# List the column names
print(df2.columns)


Total number of rows: 11847
Index(['sku', 'sku_id', 'brand_n', 'multi', 'size', 'W', 'totrev', 'qt',
       'price', 'region', 'type'],
      dtype='object')


In [53]:
#Unique identifier
unique_counts = df2.nunique()
uniqueness_ratio = (unique_counts / total_rows) * 100

# Print columns that have 100% uniqueness
potential_ids = uniqueness_ratio[uniqueness_ratio == 100].index.tolist()

print("\n--- Uniqueness Report ---")
print(uniqueness_ratio.sort_values(ascending=False).to_string())

if potential_ids:
    print(f"\nPotential Unique Identifier(s) (100% Unique): **{potential_ids}**")
else:
    print("\nNo single column is 100% unique.")


--- Uniqueness Report ---
totrev     6.170338
qt         1.561577
W          1.544695
sku        0.143496
sku_id     0.143496
brand_n    0.143496
price      0.109732
region     0.109732
multi      0.008441
size       0.008441
type       0.008441

No single column is 100% unique.


In [54]:


# 1. Define the candidates
cols_to_check = ['brand_n', 'region', 'W']

# 2. Check for uniqueness
# Group by these 3 columns and count size
duplicate_check = df1.groupby(cols_to_check).size()

# 3. Print results
print(f"Checking combination: {cols_to_check}")
if duplicate_check.max() == 1:
    print("✅ Success! ['brand_n', 'region', 'W'] is the unique identifier.")
else:
    print(f"❌ Not unique. The maximum number of repeats for a combination is {duplicate_check.max()}.")
    
    # Optional: See which ones are duplicated
    print("\nSample duplicates:")
    print(df2[df2.duplicated(subset=cols_to_check, keep=False)].sort_values(by=cols_to_check).head())

Checking combination: ['brand_n', 'region', 'W']
✅ Success! ['brand_n', 'region', 'W'] is the unique identifier.


Variables with missing data

In [55]:
# Checking for missing variables
missing_counts = df2.isnull().sum()
print("--- Count of Missing Values Per Column ---")
print(missing_counts)

# Filter the series to only include counts > 0
missing_data_columns = missing_counts[missing_counts > 0].sort_values(ascending=False)

if missing_data_columns.empty:
    print("\n✅ Great news! No missing values found in your entire dataset.")
else:
    print("\n--- Columns with Missing Values (Count) ---")
    print(missing_data_columns.to_string())

--- Count of Missing Values Per Column ---
sku            0
sku_id         0
brand_n        0
multi          0
size           0
W              0
totrev     10293
qt         10293
price      10293
region         0
type           0
dtype: int64

--- Columns with Missing Values (Count) ---
totrev    10293
qt        10293
price     10293


Making the panel balanced

In [56]:
# 1. Get the unique count for each key variable
num_brands = df2['brand_n'].nunique()
num_regions = df2['region'].nunique()
num_weeks = df2['W'].nunique()

# 2. Calculate the expected number of rows for a balanced panel
expected_rows = num_brands * num_regions * num_weeks

print(f"Unique Brands: {num_brands}")
print(f"Unique Regions: {num_regions}")
print(f"Unique Weeks (W): {num_weeks}")
print(f"Expected Rows for a BALANCED panel: {expected_rows}")

actual_rows = len(df2)
print(f"Actual Rows in df2: {actual_rows}")

if actual_rows == expected_rows:
    print("\n✅ The panel is **BALANCED**. Every brand-region combination has data for every week.")
else:
    missing_rows = expected_rows - actual_rows
    print(f"\n❌ The panel is **UNBALANCED**. There are {missing_rows} missing brand-region-week combinations.")
    print("The actual number of rows is less than the expected number.")


if actual_rows != expected_rows:
    
    # 1. Get all unique values for each key
    all_brands = df2['brand_n'].unique()
    all_regions = df2['region'].unique()
    all_weeks = df2['W'].unique()

    # 2. Create a DataFrame of ALL POSSIBLE combinations (the full panel structure)
    full_index = pd.DataFrame(
        product(all_brands, all_regions, all_weeks),
        columns=['brand_n', 'region', 'W']
    )

    # 3. Merge the full index with the actual data (using a left join)
    # The 'indicator=True' option tells us which rows only exist in the full_index
    merged_df = full_index.merge(
        df2,
        on=['brand_n', 'region', 'W'],
        how='left',
        indicator=True
    )

    # 4. Filter for rows that are missing in the original data ('left_only')
    missing_combinations = merged_df[merged_df['_merge'] == 'left_only'][['brand_n', 'region', 'W']]

    print("\n--- Missing Combinations Sample ---")
    if not missing_combinations.empty:
        # Display the first few missing combinations
        print(f"Total missing combinations found: {len(missing_combinations)}")
        print(missing_combinations.head())
    else:
        print("No missing combinations found (This should match the initial check).")

Unique Brands: 17
Unique Regions: 13
Unique Weeks (W): 183
Expected Rows for a BALANCED panel: 40443
Actual Rows in df2: 11847

❌ The panel is **UNBALANCED**. There are 28596 missing brand-region-week combinations.
The actual number of rows is less than the expected number.

--- Missing Combinations Sample ---
Total missing combinations found: 28596
    brand_n  region   W
77     1330       1  78
78     1330       1  79
79     1330       1  80
80     1330       1  81
81     1330       1  82


We donot have balanced panel, lets make one:

In [57]:


# 1. AGGREGATE DATA (With NaN protection)
# We use a lambda function to pass 'min_count=1'. 
# This ensures that if the original data was NaN, the sum remains NaN (instead of becoming 0).
df_aggregated = df2.groupby(['brand_n', 'region', 'W']).agg({
    'qt': lambda x: x.sum(min_count=1),      
    'totrev': lambda x: x.sum(min_count=1),  
    'price': 'mean', 
    'type': 'min'  
}).reset_index()

# 2. DEFINE THE SKELETON
unique_brands = df_aggregated['brand_n'].unique()
unique_regions = df_aggregated['region'].unique()
unique_weeks = df_aggregated['W'].unique()

full_index = pd.MultiIndex.from_product(
    [unique_brands, unique_regions, unique_weeks], 
    names=['brand_n', 'region', 'W']
)

# 3. REINDEX (Add the new missing rows)
# Original rows keep their data (including original NaNs preserved above).
# New rows are created as NaN.
df2_balanced = df_aggregated.set_index(['brand_n', 'region', 'W']).reindex(full_index).reset_index()

# 4. CLEANUP
cols_to_drop = ['sku', 'sku_id', 'multi', 'size']
df2_balanced = df2_balanced.drop(columns=cols_to_drop, errors='ignore')

# Fill only 'type' as requested
df2_balanced['type'] = df2_balanced['type'].fillna(2)

# --- VERIFICATION ---
print(f"Final Row Count: {len(df2_balanced)}")
print("\n--- Missing Value Counts (Includes both original and new NaNs) ---")
print(df2_balanced[['qt', 'totrev', 'price']].isnull().sum())

df2_balanced

Final Row Count: 40443

--- Missing Value Counts (Includes both original and new NaNs) ---
qt        38889
totrev    38889
price     38889
dtype: int64


,brand_n,region,W,qt,totrev,price,type
0,1329,2,1,NaN,NaN,NaN,2.0
1,1329,2,2,NaN,NaN,NaN,2.0
2,1329,2,3,NaN,NaN,NaN,2.0
3,1329,2,4,NaN,NaN,NaN,2.0
4,1329,2,5,NaN,NaN,NaN,2.0
...,...,...,...,...,...,...,...
40438,1345,8,179,NaN,NaN,NaN,2.0
40439,1345,8,180,NaN,NaN,NaN,2.0
40440,1345,8,181,NaN,NaN,NaN,2.0
40441,1345,8,182,NaN,NaN,NaN,2.0


In [58]:

def check_data_sufficiency(df, dataset_name):
    print(f"\n====== CHECKING: {dataset_name} ======")
    
    # 1. Group by Region and Week
    # We count non-null values. If count is 0, it means ALL brands are NaN for that week.
    coverage = df.groupby(['region', 'W'])[['price', 'qt', 'totrev']].count()
    
    # 2. Identify Region-Weeks with ZERO valid data
    # (i.e., The region existed, but every single brand had missing values)
    empty_price = coverage[coverage['price'] == 0]
    empty_qt = coverage[coverage['qt'] == 0]
    empty_rev = coverage[coverage['totrev'] == 0]
    
    # 3. Report Results
    total_rw = len(coverage)
    print(f"Total Region-Week combinations: {total_rw}")
    
    if empty_price.empty and empty_qt.empty and empty_rev.empty:
        print(f"✅ PASSED: All {total_rw} Region-Weeks have at least one valid observation.")
    else:
        print("❌ FAILED: Some Region-Weeks have NO valid data (all brands are NaN).")
        
        if not empty_price.empty:
            print(f"  - Missing Price in {len(empty_price)} Region-Weeks.")
            # Show first 5 missing
            print(f"    Sample: {empty_price.index.tolist()[:5]}")
            
        if not empty_qt.empty:
            print(f"  - Missing Quantity in {len(empty_qt)} Region-Weeks.")
            
        if not empty_rev.empty:
            print(f"  - Missing Revenue in {len(empty_rev)} Region-Weeks.")

# --- RUN THE CHECK ---

# 1. Check your balanced Smoke dataset
check_data_sufficiency(df1_balanced, "Cut Dataset (df1)")

# 2. Check your Heated Tobacco dataset
# (Replace 'df2' with the actual name of your heated tobacco dataframe)
# If you haven't loaded it yet, ensure you load it before running this line.
try:
    check_data_sufficiency(df2_balanced, "Smokeless Dataset (df2)") 
except NameError:
    print("\n⚠️ Note: 'df2' is not defined yet. Please load your Smokeless dataset to check it.")


====== CHECKING: Cut Dataset (df1) ======
Total Region-Week combinations: 2379
❌ FAILED: Some Region-Weeks have NO valid data (all brands are NaN).
  - Missing Price in 1781 Region-Weeks.
    Sample: [(1, 2), (1, 3), (1, 7), (1, 8), (1, 10)]
  - Missing Quantity in 1781 Region-Weeks.
  - Missing Revenue in 1781 Region-Weeks.

====== CHECKING: Smokeless Dataset (df2) ======
Total Region-Week combinations: 2379
❌ FAILED: Some Region-Weeks have NO valid data (all brands are NaN).
  - Missing Price in 1554 Region-Weeks.
    Sample: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]
  - Missing Quantity in 1554 Region-Weeks.
  - Missing Revenue in 1554 Region-Weeks.
